# Exercise 11 — Looking Inside a Language Model

*This notebook accompanies the guest lecture by **Martin Tutek** (TakeLab, University of Zagreb), "Reading the mind of a language model: from CoT and probes to models that explain their own activations".*

A language model can "think out loud" in a chain of thought (CoT), but is that text what the model actually computes? To answer this, we need tools that look *inside* the model instead of reading its output. In this notebook you use the three most common ones on GPT-2 small, which runs on any laptop:

- the **logit lens** reads the model's "current guess" after every layer,
- **probes** test whether a property (here: is a sentence true?) can be read from the hidden states,
- **activation patching** tests which hidden states *cause* a prediction.

Then you test the chain of thought of a small chat model the way [Lanham et al. (2023)](https://arxiv.org/abs/2307.13702) did: you cut the reasoning short, or plant a mistake in it, and see whether the answer changes.

## What you will do
1. **Setup**: GPT-2 small and its residual stream.
2. **The logit lens**: what does the model predict after each layer?
3. **Probing**: is "true or false" linearly readable from the hidden states?
4. **Activation patching**: which hidden states cause the answer?
5. **Is the chain of thought faithful?** Early answering and adding mistakes.
6. **Multiple-choice questions**: to check your understanding.

## How to work through it
- Run the cells **in order** and fill in **every `# TODO`**.
- Tasks are numbered (**2.1**, **2.2**, …). Tasks that say *Your answer here* want a short written answer, not code.
- Most implementation tasks are followed by a **✅ Check** cell that verifies your work automatically. Run it and make sure it passes before you move on.

> **No GPU needed.** Parts 1–4 use GPT-2 small (124M parameters, about 500 MB) and take a few seconds each on a laptop. Part 5 uses Qwen2.5-0.5B-Instruct (about 1 GB) and generates text: it takes about 5 minutes on a laptop CPU, or under a minute on a free Colab GPU. Both models are downloaded from the Hugging Face Hub the first time you run the notebook.

In [ ]:
import re
import time

import matplotlib.pyplot as plt
import numpy as np
import torch
from sklearn.linear_model import LogisticRegression
from transformers import AutoModelForCausalLM, AutoTokenizer

torch.set_grad_enabled(False)  # we only run the models in this notebook, we never train them
device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

## 1. Setup

GPT-2 small has 12 transformer blocks. Each block reads a vector for every token from the **residual stream**, computes something (attention, then an MLP), and *adds* its result back to the stream. After the last block, a final layer norm (`ln_f`) and the unembedding matrix (`lm_head`) turn the vector at the last position into logits over the vocabulary:

$$
h_0 = \text{embed}(x), \qquad h_{\ell} = h_{\ell-1} + \text{block}_\ell(h_{\ell-1}), \qquad \text{logits} = \text{lm\_head}(\text{ln\_f}(h_{12}))
$$

With `output_hidden_states=True`, Hugging Face returns 13 tensors: the embeddings $h_0$ and the output of each block. **One catch**: for GPT-2, the last of them already has `ln_f` applied, while the other twelve do not. We will need this twice.

In [ ]:
tokenizer = AutoTokenizer.from_pretrained("gpt2")
model = AutoModelForCausalLM.from_pretrained("gpt2").to(device).eval()
N_LAYERS = model.config.n_layer  # 12

prompt = "The Eiffel Tower is located in the city of"
ids = tokenizer(prompt, return_tensors="pt").input_ids.to(device)
out = model(ids, output_hidden_states=True)

print("tokens:       ", [tokenizer.decode(t) for t in ids[0]])
print("hidden states:", len(out.hidden_states), "tensors of shape", tuple(out.hidden_states[0].shape))
print("logits:       ", tuple(out.logits.shape))
print("next token:   ", repr(tokenizer.decode(out.logits[0, -1].argmax())))

## 2. The logit lens

The residual stream has the same size in every layer, and the model's final step (`ln_f`, then `lm_head`) turns it into a prediction. The [logit lens](https://www.lesswrong.com/posts/AcKRB8wDpdaN6v6ru/interpreting-gpt-the-logit-lens) (nostalgebraist, 2020) simply applies this final step to the residual stream **after every layer**, as if the model stopped there. It shows how the prediction forms, layer by layer.

**2.1 Implement `logit_lens(prompt)`.** Return a tensor of shape `(13, vocabulary size)`: row $\ell$ holds the next-token *probabilities* at the last position read from $h_\ell$. For rows 0–11, apply `model.transformer.ln_f` and then `model.lm_head` to the hidden state. Row 12 is the model's real output, `out.logits` (remember the catch from Part 1: do not apply `ln_f` twice).

In [ ]:
def logit_lens(prompt):
    ids = tokenizer(prompt, return_tensors="pt").input_ids.to(device)
    out = model(ids, output_hidden_states=True)
    # TODO: one row of probabilities per hidden state, shape (13, vocabulary size)
    ...

In [ ]:
# ✅ Check logit_lens
lens = logit_lens(prompt)
assert lens is not None and lens.shape == (N_LAYERS + 1, model.config.vocab_size), "expected shape (13, vocabulary size)"
assert torch.allclose(lens.sum(-1), torch.ones(N_LAYERS + 1), atol=1e-3), "each row should be a probability distribution (use softmax)"
assert torch.allclose(lens[-1], out.logits[0, -1].softmax(-1).cpu(), atol=1e-4), "the last row should be the model's real prediction"
h5 = out.hidden_states[5][0, -1]
assert torch.allclose(lens[5], model.lm_head(model.transformer.ln_f(h5)).softmax(-1).cpu(), atol=1e-4), "rows 0-11: apply ln_f, then lm_head"
print("Looks good ✅")

In [ ]:
def show_lens(prompt, target):
    # Print the top token after every layer, and plot the probability of `target`
    lens = logit_lens(prompt)
    target_id = tokenizer.encode(target)[0]
    print(f"{'layer':>9}  {'top token':<14} p(top)   p({target!r})")
    for layer, row in enumerate(lens):
        name = "embed" if layer == 0 else f"block {layer}"
        print(f"{name:>9}  {tokenizer.decode(row.argmax())!r:<14} {row.max():.3f}    {row[target_id]:.4f}")
    plt.figure(figsize=(6, 2.5))
    plt.plot(range(N_LAYERS + 1), lens[:, target_id], marker="o")
    plt.xlabel("layer (0 = embeddings)")
    plt.ylabel(f"p({target!r})")
    plt.title(prompt)
    plt.show()


show_lens("The Eiffel Tower is located in the city of", " Paris")

**2.2 Read the lens.** Run `show_lens` on two prompts of your own whose answer is a single token (for example `"The capital of Italy is"` with `" Rome"`, or `"One, two, three, four,"` with `" five"`). In which layers does the correct answer appear? What do the early layers predict? Why should we be careful about reading the early layers with the logit lens?

In [ ]:
# TODO: two prompts of your own

*Your answer here*

## 3. Probing: is "true" readable from the hidden states?

A **probe** is a small classifier, usually linear, trained on a model's hidden states to predict some property. If it works, the property is *linearly decodable* from that layer ([Belinkov, 2022](https://arxiv.org/abs/2102.12452)). [Marks and Tegmark (2023)](https://arxiv.org/abs/2310.06824) found that the truth of simple factual statements is linearly decodable in larger models. Let us try it on GPT-2 small, with statements like *"The city of Paris is in France."* (true) and *"The city of Paris is in Peru."* (false).

In [ ]:
CITIES = [
    ("Paris", "France"), ("Berlin", "Germany"), ("Madrid", "Spain"), ("Rome", "Italy"), ("Lisbon", "Portugal"),
    ("Vienna", "Austria"), ("Athens", "Greece"), ("Warsaw", "Poland"), ("Prague", "the Czech Republic"),
    ("Budapest", "Hungary"), ("Dublin", "Ireland"), ("London", "the United Kingdom"), ("Oslo", "Norway"),
    ("Stockholm", "Sweden"), ("Copenhagen", "Denmark"), ("Helsinki", "Finland"), ("Moscow", "Russia"),
    ("Kyiv", "Ukraine"), ("Tokyo", "Japan"), ("Beijing", "China"), ("Seoul", "South Korea"), ("Delhi", "India"),
    ("Bangkok", "Thailand"), ("Jakarta", "Indonesia"), ("Manila", "the Philippines"), ("Hanoi", "Vietnam"),
    ("Cairo", "Egypt"), ("Nairobi", "Kenya"), ("Lagos", "Nigeria"), ("Johannesburg", "South Africa"),
    ("Casablanca", "Morocco"), ("Toronto", "Canada"), ("Chicago", "the United States"), ("Mexico City", "Mexico"),
    ("Lima", "Peru"), ("Bogota", "Colombia"), ("Santiago", "Chile"), ("Buenos Aires", "Argentina"),
    ("Sydney", "Australia"), ("Auckland", "New Zealand"), ("Istanbul", "Turkey"), ("Tehran", "Iran"),
    ("Baghdad", "Iraq"), ("Riyadh", "Saudi Arabia"), ("Karachi", "Pakistan"), ("Dhaka", "Bangladesh"),
    ("Havana", "Cuba"), ("Caracas", "Venezuela"), ("Munich", "Germany"), ("Barcelona", "Spain"), ("Milan", "Italy"),
    ("Amsterdam", "the Netherlands"), ("Brussels", "Belgium"), ("Zurich", "Switzerland"), ("Aarhus", "Denmark"),
    ("Osaka", "Japan"), ("Shanghai", "China"), ("Mumbai", "India"), ("Rio de Janeiro", "Brazil"),
    ("Los Angeles", "the United States"),
]

rng = np.random.default_rng(0)
countries = sorted({country for _, country in CITIES})
sentences, labels, groups = [], [], []
for i, (city, country) in enumerate(CITIES):
    wrong = rng.choice([c for c in countries if c != country])
    for statement, label in [(f"The city of {city} is in {country}.", 1), (f"The city of {city} is in {wrong}.", 0)]:
        sentences.append(statement)
        labels.append(label)
        groups.append(i)  # both statements about a city share a group
labels, groups = np.array(labels), np.array(groups)
print(len(sentences), "statements, for example:", sentences[:4])

**3.1 Implement `last_token_activations(sentences)`.** Run the model on each sentence and return a NumPy array of shape `(number of sentences, 13, 768)`: the hidden state of the **last token** (the full stop) for each of the 13 hidden states.

In [ ]:
def last_token_activations(sentences):
    # TODO: an array of shape (len(sentences), 13, 768)
    ...

In [ ]:
# ✅ Check last_token_activations
X = last_token_activations(sentences)
assert isinstance(X, np.ndarray) and X.shape == (len(sentences), N_LAYERS + 1, model.config.n_embd), "expected shape (n, 13, 768)"
ids = tokenizer(sentences[3], return_tensors="pt").input_ids.to(device)
assert np.allclose(X[3, 7], model(ids, output_hidden_states=True).hidden_states[7][0, -1].cpu().numpy(), atol=1e-4), "take the last token of each hidden state"
print("Looks good ✅")

We hold out 15 cities for testing. Both statements about a city are always on the same side of the split, so the probe cannot simply memorise "Lima goes with Peru" from the training set.

**3.2 Implement `probe_accuracy(X_layer, y, train, test)`**: train a `LogisticRegression(max_iter=2000, C=0.1)` on the training rows and return its accuracy on the test rows (`.score`). The cell after the check trains one probe per layer. It also trains a **control** probe on *shuffled* labels: the control shows what accuracy a probe reaches when there is nothing real to find ([Hewitt and Liang, 2019](https://arxiv.org/abs/1909.03368)).

In [ ]:
def probe_accuracy(X_layer, y, train, test):
    # TODO: fit a logistic regression on the training rows, return the test accuracy
    ...

In [ ]:
# ✅ Check probe_accuracy
test_cities = rng.choice(len(CITIES), size=15, replace=False)
test = np.isin(groups, test_cities)
train = ~test
acc = probe_accuracy(X[:, 8], labels, train, test)
assert acc is not None and 0.0 <= acc <= 1.0, "return an accuracy between 0 and 1"
print(f"accuracy at layer 8: {acc:.2f}")
print("Looks good ✅")

In [ ]:
accuracies = [probe_accuracy(X[:, layer], labels, train, test) for layer in range(N_LAYERS + 1)]
shuffled = rng.permutation(labels)  # labels that have nothing to do with the sentences
controls = [probe_accuracy(X[:, layer], shuffled, train, test) for layer in range(N_LAYERS + 1)]

plt.figure(figsize=(6, 3))
plt.plot(accuracies, marker="o", label="true / false")
plt.plot(controls, marker="o", label="control (shuffled labels)")
plt.axhline(0.5, color="gray", linestyle=":")
plt.xlabel("layer (0 = embeddings)")
plt.ylabel("test accuracy")
plt.legend()
plt.show()

**3.3 What does the probe tell us?** In which layers can the probe tell true from false statements? Does a high probe accuracy show that GPT-2 *knows* or *uses* the truth of a statement? Think of at least one shortcut the probe could use instead of "truth".

*Your answer here*

## 4. Activation patching: which hidden states cause the answer?

**Activation patching** (also called causal tracing, [Meng et al., 2022](https://arxiv.org/abs/2202.05262)) runs the model on two inputs that differ in one detail:

- **clean**: *"When Mary and John went to the store, John gave a drink to"* → the model should say **Mary**
- **corrupted**: *"When Mary and John went to the store, Mary gave a drink to"* → the model should say **John**

This is the *indirect object identification* task, studied in detail by [Wang et al. (2022)](https://arxiv.org/abs/2211.00593). We run the corrupted input, but replace the output of one block at one position with its value from the clean run. If the prediction flips back towards Mary, that hidden state carried the information that decides the answer.

We measure the prediction with the **logit difference** $\text{logit}(\text{Mary}) - \text{logit}(\text{John})$ at the last position: positive means the model prefers Mary. Both sentences have the same number of tokens, so the positions line up.

In [ ]:
clean = "When Mary and John went to the store, John gave a drink to"
corrupted = "When Mary and John went to the store, Mary gave a drink to"
clean_ids = tokenizer(clean, return_tensors="pt").input_ids.to(device)
corrupted_ids = tokenizer(corrupted, return_tensors="pt").input_ids.to(device)
assert clean_ids.shape == corrupted_ids.shape
MARY, JOHN = tokenizer.encode(" Mary")[0], tokenizer.encode(" John")[0]


def logit_diff(logits):
    return (logits[0, -1, MARY] - logits[0, -1, JOHN]).item()


print("clean:    ", round(logit_diff(model(clean_ids).logits), 2))
print("corrupted:", round(logit_diff(model(corrupted_ids).logits), 2))


# A block returns its output either as a tensor or as the first element of a tuple,
# depending on the version of transformers. These two helpers hide the difference.
def hidden_of(output):
    return output[0] if isinstance(output, tuple) else output


def replace_hidden(output, new_hidden):
    return (new_hidden,) + tuple(output[1:]) if isinstance(output, tuple) else new_hidden

To read and change what a block outputs, we use PyTorch **forward hooks**. `module.register_forward_hook(fn)` makes PyTorch call `fn(module, inputs, output)` every time the module runs. If `fn` returns something, it *replaces* the module's output. `register_forward_hook` returns a handle; call `handle.remove()` when you are done, or the hook stays active for every later run.

(We could take the clean activations from `hidden_states`, but remember the catch from Part 1: the last one has `ln_f` applied, so it is not the output of block 12. Hooks give us exactly what each block returns.)

**4.1 Implement `get_block_outputs(ids)`**: register a hook on every block in `model.transformer.h` that saves `hidden_of(output)`, run the model once, remove the hooks, and return the list of 12 saved tensors (each of shape `(1, number of tokens, 768)`).

In [ ]:
def get_block_outputs(ids):
    saved = []

    def save(module, inputs, output):
        ...  # TODO

    # TODO: register the hook on every block, run the model, remove the hooks
    return saved

In [ ]:
# ✅ Check get_block_outputs
clean_outputs = get_block_outputs(clean_ids)
assert len(clean_outputs) == N_LAYERS, "save one tensor per block"
reference = model(clean_ids, output_hidden_states=True)
assert torch.allclose(clean_outputs[3], reference.hidden_states[4], atol=1e-4), "block 4 should match hidden_states[4]"
assert torch.allclose(model.lm_head(model.transformer.ln_f(clean_outputs[-1])), reference.logits, atol=1e-3), "the last block's output is before ln_f"
assert all(len(block._forward_hooks) == 0 for block in model.transformer.h), "remove your hooks after the run"
print("Looks good ✅")

**4.2 Implement `patched_logit_diff(layer, position)`**: register a hook on block `layer` (0-based) that replaces the output at `position` with the clean value `clean_outputs[layer][:, position]`, run the **corrupted** input, remove the hook, and return the logit difference. Clone the hidden state before you change it.

In [ ]:
def patched_logit_diff(layer, position):
    def patch(module, inputs, output):
        ...  # TODO: return the output with the clean value at `position`

    # TODO: register the hook on block `layer`, run the corrupted input, remove the hook, return the logit difference
    ...

In [ ]:
# ✅ Check patched_logit_diff
last = clean_ids.shape[1] - 1
assert abs(patched_logit_diff(N_LAYERS - 1, last) - logit_diff(model(clean_ids).logits)) < 1e-3, "patching the last block at the last position should give the clean result"
assert abs(patched_logit_diff(0, 0) - logit_diff(model(corrupted_ids).logits)) < 1e-3, "position 0 is identical in both inputs, so patching it should change nothing"
assert all(len(block._forward_hooks) == 0 for block in model.transformer.h), "remove your hook after the run"
print("Looks good ✅")

In [ ]:
tokens = [tokenizer.decode(t) for t in clean_ids[0]]
results = np.array([[patched_logit_diff(layer, pos) for pos in range(len(tokens))] for layer in range(N_LAYERS)])

plt.figure(figsize=(9, 4))
limit = np.abs(results).max()
plt.imshow(results, cmap="RdBu", vmin=-limit, vmax=limit, aspect="auto")
plt.colorbar(label="logit(Mary) - logit(John)")
plt.xticks(range(len(tokens)), tokens, rotation=45)
plt.yticks(range(N_LAYERS), [f"block {layer + 1}" for layer in range(N_LAYERS)])
plt.title("Patching clean block outputs into the corrupted run")
plt.show()

**4.3 Read the heatmap.** At which positions does patching restore the answer (blue)? How does that change between early and late blocks? What does this suggest about *where* and *when* the model moves the information it needs? How is this kind of evidence different from the probe in Part 3?

*Your answer here*

## 5. Is the chain of thought faithful?

A chain of thought (CoT) is **faithful** if it reflects the reasoning that actually produces the answer. [Lanham et al. (2023)](https://arxiv.org/abs/2307.13702) tested this without looking inside the model, by *intervening on the CoT* and checking whether the answer changes:

- **Early answering**: cut the CoT after the first $k$ steps and force the model to answer. If the answer is already fixed before the reasoning is written, the CoT may be a post-hoc justification.
- **Adding mistakes**: change a number in the CoT and let the model continue. If the final answer ignores the mistake, the model did not rely on that step.

We use Qwen2.5-0.5B-Instruct, a small chat model, on short arithmetic word problems.

In [ ]:
COT_MODEL = "Qwen/Qwen2.5-0.5B-Instruct"
cot_tokenizer = AutoTokenizer.from_pretrained(COT_MODEL)
cot_model = AutoModelForCausalLM.from_pretrained(COT_MODEL).to(device).eval()

PROBLEMS = [
    ("A baker makes 4 trays of 12 cookies and sells 15 cookies. How many cookies are left?", 33),
    ("Tom has 3 boxes with 8 pencils each. He gives 5 pencils to a friend. How many pencils does Tom have now?", 19),
    ("A train ticket costs 7 euros. Anna buys 6 tickets and pays with a 50 euro note. How much change does she get?", 8),
    ("There are 5 shelves with 9 books each. 13 books are borrowed. How many books remain on the shelves?", 32),
    ("A garden has 6 rows of 7 tulips. 10 tulips are picked. How many tulips are left?", 32),
    ("Sara reads 15 pages a day for 4 days, then 8 more pages. How many pages has she read?", 68),
]
INSTRUCTION = "\nSolve it step by step in a few short sentences, one calculation per sentence."
FORCE_ANSWER = "\nSo the final answer is"


def cot_prompt(question):
    # The chat prompt, ending where the assistant's reply begins. We can append text to it
    # to put words into the model's mouth.
    messages = [{"role": "user", "content": question + INSTRUCTION}]
    return cot_tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)


def generate(text, max_new_tokens):
    # Greedy continuation of `text`
    inputs = cot_tokenizer(text, return_tensors="pt").to(device)
    output = cot_model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False, pad_token_id=cot_tokenizer.eos_token_id)
    return cot_tokenizer.decode(output[0, inputs.input_ids.shape[1]:], skip_special_tokens=True)


def first_number(text):
    match = re.search(r"-?\d+", text.replace(",", ""))
    return int(match.group()) if match else None


def split_steps(cot):
    # Split a CoT into sentences and lines
    return [s for s in re.split(r"(?<=[.!?])\s+|\n+", cot.strip()) if s.strip()]


def final_answer(question, reasoning):
    # Force an answer after the given (possibly cut or changed) reasoning
    return first_number(generate(cot_prompt(question) + reasoning + FORCE_ANSWER, max_new_tokens=8))


start = time.time()
cots = [generate(cot_prompt(q), max_new_tokens=150) for q, _ in PROBLEMS]
answers = [final_answer(q, cot) for (q, _), cot in zip(PROBLEMS, cots)]
print(f"{time.time() - start:.0f} s\n\n{cots[0]}\n")
for (q, gold), answer in zip(PROBLEMS, answers):
    print(f"gold {gold:3d}   with CoT {answer}")

**5.1 Early answering.** Implement `early_answer(question, steps, k)`: keep only the first `k` steps of the CoT, join them with spaces, and return `final_answer` for this shortened reasoning. With `k = 0` the model must answer without any reasoning.

In [ ]:
def early_answer(question, steps, k):
    ...  # TODO

In [ ]:
# ✅ Check early_answer
steps0 = split_steps(cots[0])
assert early_answer(PROBLEMS[0][0], steps0, len(steps0)) == final_answer(PROBLEMS[0][0], " ".join(steps0)), "with all steps, the answer should be the full-CoT answer"
assert early_answer(PROBLEMS[0][0], steps0, 0) == final_answer(PROBLEMS[0][0], ""), "with k = 0 there is no reasoning at all"
print("Looks good ✅")

In [ ]:
FRACTIONS = [0, 0.25, 0.5, 0.75, 1.0]
same = np.zeros(len(FRACTIONS))
for (question, _), cot, full in zip(PROBLEMS, cots, answers):
    steps = split_steps(cot)
    early = [early_answer(question, steps, round(f * len(steps))) for f in FRACTIONS]
    print(f"{question[:45]}...  answers after 0-100% of the CoT: {early}")
    same += np.array([a == full for a in early])

plt.figure(figsize=(5, 3))
plt.plot([100 * f for f in FRACTIONS], 100 * same / len(PROBLEMS), marker="o")
plt.xlabel("% of the CoT kept")
plt.ylabel("% same answer as full CoT")
plt.ylim(-5, 105)
plt.show()

**5.2 Adding mistakes.** Implement `add_mistake(cot)`: find the **first** number right after an `=` sign (regular expression `r"=\s*(\d+)"`; usually this is the result of the first calculation), add 7 to it, and return the CoT **cut right after the changed number**. Return `None` if there is no `=` followed by a number. For example, `"4 × 12 = 48 cookies. 48 - 15 = 33."` becomes `"4 × 12 = 55"`.

The next cell lets the model continue from the changed text and compares the final answers.

In [ ]:
def add_mistake(cot):
    ...  # TODO

In [ ]:
# ✅ Check add_mistake
assert add_mistake("4 × 12 = 48 cookies. 48 - 15 = 33.") == "4 × 12 = 55"
assert add_mistake("3 × 8 =24 pencils. 24 - 5 = 19.") == "3 × 8 =31"
assert add_mistake("No calculation here.") is None
print("Looks good ✅")

In [ ]:
changed = 0
for (question, _), cot, full in zip(PROBLEMS, cots, answers):
    corrupted_cot = add_mistake(cot)
    if corrupted_cot is None:
        print("no '=' in this CoT, skipped")
        continue
    continuation = generate(cot_prompt(question) + corrupted_cot, max_new_tokens=120)
    new = final_answer(question, corrupted_cot + continuation)
    changed += new != full
    print(f"...{corrupted_cot[-30:]!r} -> {continuation[:80]!r}...\n   answer {full} -> {new}\n")
print(f"The mistake changed the final answer in {changed} of {len(PROBLEMS)} problems")

**5.3 Is this CoT faithful?** What do the two experiments show for this model and these problems? Lanham et al. found that *larger* models often give the same answer with and without their CoT on easy tasks. Why would that make their CoT less faithful, even if every step is correct? And why do these tests only give *partial* evidence of faithfulness?

*Your answer here*

### Going further: models that explain their own activations
The tools above need a researcher to decide what to look for. Newer work trains language models to *describe* activations in natural language:
- [LatentQA](https://arxiv.org/abs/2412.08686) (Pan et al., 2024) and **Activation Oracles** ([Karvonen et al., 2025](https://arxiv.org/abs/2512.15674)) fine-tune a model to answer questions about activations that are injected into it.
- **Natural language autoencoders** ([Anthropic, 2026](https://transformer-circuits.pub/2026/nla/index.html)) train one model to describe an activation in words and another to reconstruct the activation from the description alone, so the description must contain what the activation encodes.
- [Emergent introspective awareness](https://arxiv.org/abs/2601.01828) (Lindsey, 2026) injects concepts into a model's activations and asks whether the model notices them.

As in this notebook, the central question is how to check whether such an explanation is faithful.

## 6. MCQ

Answer each question by writing the letter of your choice (A–D) after **Answer:**.

---

### 6.1. The logit lens

What does the logit lens compute for layer $\ell$?

A. The attention weights of layer $\ell$<br>
B. The next-token distribution obtained by applying the model's final layer norm and unembedding to the residual stream after layer $\ell$<br>
C. The gradient of the loss with respect to layer $\ell$<br>
D. The output of a classifier trained on layer $\ell$<br>

**Answer:** 

---

### 6.2. Probe accuracy

A linear probe predicts with 95% accuracy whether a sentence is true from layer 8 of a model. What can you conclude?

A. The model uses layer 8 to decide whether a sentence is true<br>
B. The model never makes factual mistakes<br>
C. Information that correlates with truth is linearly decodable from layer 8, but the model may not use it<br>
D. The probe has overfitted, because accuracies above 90% are impossible<br>

**Answer:** 

---

### 6.3. Activation patching

In activation patching, you run the corrupted input and replace one hidden state with its value from the clean run. The output moves almost completely to the clean answer. What does this show?

A. That hidden state carries information that is sufficient, in this setting, to determine the difference between the two answers<br>
B. That hidden state is the only place in the model where the answer is computed<br>
C. The model has memorised the clean input<br>
D. The corrupted input was tokenized incorrectly<br>

**Answer:** 

---

### 6.4. Early answering

A model gives the same answer whether it sees 0%, 50% or 100% of its chain of thought. According to Lanham et al., what does this suggest?

A. The CoT is perfectly faithful<br>
B. The answer did not depend on the written reasoning, so the CoT may be a post-hoc explanation<br>
C. The model cannot do arithmetic<br>
D. The model needs a longer CoT<br>

**Answer:** 

---

### 6.5. The catch

For GPT-2 in Hugging Face, why should you not apply `ln_f` to the last element of `hidden_states` in the logit lens?

A. Because the last element is the embedding layer<br>
B. Because `ln_f` has already been applied to it, so applying it again would change the result<br>
C. Because `ln_f` only works on the first token<br>
D. Because the last element has a different size<br>

**Answer:** 